# Project Lab 2, A Library From Scratch, a Validation Study and Customer Segments

Notebook for **Chapter 18** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch18.html

Tip: change a number, predict the output, then run the cell again.

## 18.1 Project: A From-Scratch ML Library in NumPy, Tested Against scikit-learn

In [ ]:
"""Tiny ML library in NumPy. Educational, not fast."""
import numpy as np

class LinearRegression:
    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]
        self.coef_ = np.linalg.lstsq(A, y, rcond=None)[0]
        return self
    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.coef_

class LogisticRegression:
    """Newton's method (IRLS) with a small L2 penalty on the weights (not the intercept)."""
    def __init__(self, C=1e6, n_iter=50):
        self.lam, self.n_iter = 1.0 / C, n_iter
    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]; w = np.zeros(A.shape[1])
        R = self.lam * np.eye(A.shape[1]); R[0, 0] = 0
        for _ in range(self.n_iter):
            p = 1 / (1 + np.exp(-A @ w))
            grad = A.T @ (p - y) + R @ w
            H = A.T @ (A * (p * (1 - p))[:, None]) + R
            step = np.linalg.solve(H, grad); w -= step
            if np.abs(step).max() < 1e-10: break
        self.w_ = w; return self
    def predict_proba(self, X):
        p = 1 / (1 + np.exp(-(np.c_[np.ones(len(X)), X] @ self.w_)))
        return np.c_[1 - p, p]
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= .5).astype(int)

class KNN:
    def __init__(self, k=5): self.k = k
    def fit(self, X, y): self.X_, self.y_ = np.asarray(X, float), np.asarray(y); return self
    def predict(self, X):
        d = ((np.asarray(X, float)[:, None, :] - self.X_[None, :, :]) ** 2).sum(-1)
        idx = np.argsort(d, axis=1, kind="stable")[:, :self.k]
        votes = self.y_[idx]
        return np.array([np.bincount(v).argmax() for v in votes])

class DecisionTree:
    def __init__(self, max_depth=3, min_leaf=1): self.max_depth, self.min_leaf = max_depth, min_leaf
    @staticmethod
    def _gini(y): 
        p = np.bincount(y) / len(y); return 1 - (p ** 2).sum()
    def _best(self, X, y):
        best = (None, None, self._gini(y)); n = len(y)
        for j in range(X.shape[1]):
            vals = np.unique(X[:, j])
            for t in (vals[:-1] + vals[1:]) / 2:
                L = X[:, j] <= t
                if L.sum() < self.min_leaf or (~L).sum() < self.min_leaf: continue
                g = (L.sum() * self._gini(y[L]) + (~L).sum() * self._gini(y[~L])) / n
                if g < best[2] - 1e-12: best = (j, t, g)
        return best
    def _grow(self, X, y, depth):
        if depth == self.max_depth or len(np.unique(y)) == 1: return int(np.bincount(y).argmax())
        j, t, _ = self._best(X, y)
        if j is None: return int(np.bincount(y).argmax())
        L = X[:, j] <= t
        return (j, t, self._grow(X[L], y[L], depth + 1), self._grow(X[~L], y[~L], depth + 1))
    def fit(self, X, y): self.tree_ = self._grow(np.asarray(X, float), np.asarray(y), 0); return self
    def _one(self, x, node):
        while isinstance(node, tuple): node = node[2] if x[node[0]] <= node[1] else node[3]
        return node
    def predict(self, X): return np.array([self._one(x, self.tree_) for x in np.asarray(X, float)])

class KMeans:
    def __init__(self, k=3, init=None, n_iter=100, seed=0): self.k, self.init, self.n_iter, self.seed = k, init, n_iter, seed
    def fit(self, X):
        X = np.asarray(X, float)
        if self.init is None:
            rng = np.random.default_rng(self.seed); C = X[rng.choice(len(X), self.k, replace=False)]
        else: C = np.array(self.init, float)
        for _ in range(self.n_iter):
            lab = ((X[:, None, :] - C[None]) ** 2).sum(-1).argmin(1)
            newC = np.array([X[lab == c].mean(0) if (lab == c).any() else C[c] for c in range(self.k)])
            if np.allclose(newC, C): break
            C = newC
        self.centers_, self.labels_ = C, lab
        self.inertia_ = ((X - C[lab]) ** 2).sum()
        return self
    def predict(self, X): return ((np.asarray(X, float)[:, None, :] - self.centers_[None]) ** 2).sum(-1).argmin(1)

In [ ]:
import numpy as np
from sklearn.datasets import make_regression, make_classification, make_blobs
from sklearn import linear_model, neighbors, tree, cluster
import mlscratch as ms

def test_linear():
    X, y = make_regression(200, 4, noise=5, random_state=0)
    a = ms.LinearRegression().fit(X, y); b = linear_model.LinearRegression().fit(X, y)
    assert np.allclose(a.coef_[1:], b.coef_) and np.isclose(a.coef_[0], b.intercept_)
    assert np.allclose(a.predict(X), b.predict(X))

def test_logistic():
    X, y = make_classification(300, 5, random_state=1)
    a = ms.LogisticRegression(C=1.0).fit(X, y); b = linear_model.LogisticRegression(C=1.0, tol=1e-10, max_iter=1000).fit(X, y)
    assert np.allclose(a.w_[1:], b.coef_[0], atol=1e-4) and np.isclose(a.w_[0], b.intercept_[0], atol=1e-4)
    assert np.allclose(a.predict_proba(X)[:, 1], b.predict_proba(X)[:, 1], atol=1e-4)

def test_knn():
    X, y = make_classification(200, 4, random_state=2); Xt = X[:50] + .01
    a = ms.KNN(5).fit(X, y).predict(Xt); b = neighbors.KNeighborsClassifier(5).fit(X, y).predict(Xt)
    assert (a == b).all()

def test_tree():
    X, y = make_classification(200, 4, random_state=3)
    a = ms.DecisionTree(max_depth=3).fit(X, y); b = tree.DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
    agree = (a.predict(X) == b.predict(X)).mean(); print("tree agreement", agree)
    assert agree >= 0.99

def test_kmeans():
    X, _ = make_blobs(300, centers=3, random_state=4)
    init = X[[0, 1, 2]]
    a = ms.KMeans(3, init=init).fit(X); b = cluster.KMeans(3, init=init, n_init=1, algorithm="lloyd").fit(X)
    assert np.allclose(a.centers_, b.cluster_centers_, atol=1e-6) and np.isclose(a.inertia_, b.inertia_)

def test_logistic_shape_and_range():
    X, y = make_classification(50, 3, n_informative=2, n_redundant=0, random_state=5); p = ms.LogisticRegression().fit(X, y).predict_proba(X)
    assert p.shape == (50, 2) and np.allclose(p.sum(1), 1) and (p >= 0).all()

if __name__ == "__main__":      # tiny runner, so the file also works without pytest
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            fn(); print("PASS", name)

## 18.2 Project: A Validation and Hyperparameter-Tuning Study That Shows Leakage

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score, StratifiedKFold, ParameterSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# ---------- Study A: split strategy ----------
def make_visits(n_cust=400, per=12, seed=0):
    rng = np.random.default_rng(seed); rows = []
    for c in range(n_cust):
        quirk = rng.normal(0, 1.5, 3); base = rng.normal(0, 2)      # customer-specific habit (identifies the customer)
        for t in range(per):
            x = quirk + rng.normal(0, .15, 3)
            w1 = 1.0 - 0.25*t                                    # relationship changes with time
            z = base + w1*x[1]
            rows.append((c, t, *x, rng.binomial(1, 1/(1+np.exp(-z)))))
    return pd.DataFrame(rows, columns=["cust","month","x0","x1","x2","y"])
d = make_visits(); F = ["x0","x1","x2"]
model = lambda: RandomForestClassifier(150, random_state=0, n_jobs=1)
def cv(splitter, data, groups=None):
    return cross_val_score(model(), data[F], data.y, groups=groups, cv=splitter, scoring="roc_auc").mean()
# Question 1: "will it work for NEW customers?"  Truth = train on customers 0-299, score customers 300-399.
old, new = d[d.cust < 300], d[d.cust >= 300]
truth1 = roc_auc_score(new.y, model().fit(old[F], old.y).predict_proba(new[F])[:,1])
print("rows", len(d), "customers", d.cust.nunique())
print("Q1 new customers:")
print(f"  random KFold(5)        : {cv(KFold(5, shuffle=True, random_state=0), old):.3f}")
print(f"  GroupKFold(5) by cust  : {cv(GroupKFold(5), old, old.cust):.3f}")
print(f"  TRUTH (unseen customers): {truth1:.3f}")

# Question 2: "will it work next month?"  A drifting world with no repeated customers.
def make_drift(n=6000, seed=5):
    rng = np.random.default_rng(seed); t = np.sort(rng.integers(0, 12, n)); X = rng.normal(size=(n, 3))
    z = (1.0 - 0.25*t) * X[:,1] + 0.5*X[:,2]               # the weight on x1 fades and flips sign
    return pd.DataFrame(dict(month=t, x0=X[:,0], x1=X[:,1], x2=X[:,2], y=rng.binomial(1, 1/(1+np.exp(-z)))))
dd = make_drift(); past, future = dd[dd.month < 9], dd[dd.month >= 9]
truth2 = roc_auc_score(future.y, model().fit(past[F], past.y).predict_proba(future[F])[:,1])
print("Q2 next months (train months 0-8, score months 9-11):")
print(f"  random KFold(5)        : {cv(KFold(5, shuffle=True, random_state=0), past):.3f}")
print(f"  TimeSeriesSplit(4)     : {cv(TimeSeriesSplit(4), past):.3f}")
print(f"  TRUTH (future months)  : {truth2:.3f}")

# ---------- Study B: leakage in feature selection (labels are pure noise) ----------
cvs = StratifiedKFold(5, shuffle=True, random_state=0); W, R = [], []
for seed in range(20):
    rng = np.random.default_rng(seed); Xn = rng.normal(size=(100, 1000)); yn = rng.integers(0, 2, 100)
    Xsel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)          # WRONG: selection saw every row
    W.append(cross_val_score(LogisticRegression(max_iter=1000), Xsel, yn, cv=cvs, scoring="roc_auc").mean())
    R.append(cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), Xn, yn, cv=cvs, scoring="roc_auc").mean())
print(f"noise labels, mean of 20 datasets: select-then-CV (leak) AUC={np.mean(W):.3f}; selection inside pipeline AUC={np.mean(R):.3f}; truth 0.5")

# ---------- Study C: over-tuning on a small validation set (30 repeats) ----------
space = dict(max_depth=list(range(1,15)), min_samples_leaf=list(range(1,40)), max_features=[1,2,3,5,10], criterion=["gini","entropy"])
def gen(rng, n):
    X = rng.normal(size=(n, 10)); return X, (X[:,0] + 0.5*X[:,1] + rng.normal(0, 2.0, n) > 0).astype(int)
best_val, fresh_of_best, default_fresh = [], [], []
for rep in range(30):
    rng = np.random.default_rng(100+rep); Xt, yt = gen(rng, 150); Xv, yv = gen(rng, 60); Xf, yf = gen(rng, 5000)
    sc = []
    for p in ParameterSampler(space, 100, random_state=rep):
        m = DecisionTreeClassifier(random_state=0, **p).fit(Xt, yt)
        sc.append((roc_auc_score(yv, m.predict_proba(Xv)[:,1]), roc_auc_score(yf, m.predict_proba(Xf)[:,1])))
    sc = np.array(sc); b = sc[:,0].argmax(); best_val.append(sc[b,0]); fresh_of_best.append(sc[b,1])
print(f"mean best validation AUC={np.mean(best_val):.3f}; mean AUC of that same winner on 5000 fresh rows={np.mean(fresh_of_best):.3f}")

## 18.3 Project: Customer Segmentation With Clustering and Stability Analysis

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
def make_shoppers(seed=0):
    rng = np.random.default_rng(seed)
    #        name     n   recency(mean,sd)  orders/yr(mean,sd)  basket(mean,sd)
    spec = [("loyal",   500, (12, 5),   (12, 2),  (30, 4)),
            ("bargain", 700, (40, 15),  (12, 3),  (14, 3)),
            ("lapsed",  400, (160, 50), (2, 1),   (35, 10))]
    parts = [pd.DataFrame(dict(recency=rng.normal(*r, n).clip(1), orders=rng.normal(*o, n).clip(1),
                               basket=rng.normal(*m, n).clip(5), true=name)) for name, n, r, o, m in spec]
    df = pd.concat(parts, ignore_index=True)
    df["annual_spend"] = df.orders * df.basket
    return df.sample(frac=1, random_state=0).reset_index(drop=True)
df = make_shoppers(); F = ["recency","orders","basket"]
print(df.groupby("true")[F+["annual_spend"]].mean().round(1))
Xs = StandardScaler().fit_transform(df[F])
print("k | silhouette | inertia | bootstrap-ARI(mean±sd)")
def boot_stability(X, k, B=30, seed=0):
    rng = np.random.default_rng(seed); ref = KMeans(k, n_init=10, random_state=0).fit(X).labels_; aris = []
    for b in range(B):
        idx = rng.choice(len(X), len(X), replace=True)
        m = KMeans(k, n_init=10, random_state=b).fit(X[idx]); aris.append(adjusted_rand_score(ref, m.predict(X)))
    return np.mean(aris), np.std(aris)
for k in range(2, 8):
    km = KMeans(k, n_init=10, random_state=0).fit(Xs); m, s = boot_stability(Xs, k)
    print(f"{k} | {silhouette_score(Xs, km.labels_):.3f} | {km.inertia_:8.1f} | {m:.3f} ± {s:.3f}")
km = KMeans(3, n_init=10, random_state=0).fit(Xs)
print("ARI vs hidden truth:", round(adjusted_rand_score(df.true, km.labels_), 3))
df["seg"] = km.labels_
print(pd.crosstab(df.seg, df.true))
print(df.groupby("seg")[F+["annual_spend"]].agg("mean").round(1).assign(size=df.groupby("seg").size()))
# mistake: unscaled
ku = KMeans(3, n_init=10, random_state=0).fit(df[F]); print("UNSCALED ARI vs truth:", round(adjusted_rand_score(df.true, ku.labels_),3))
# seed stability with n_init=1
aris=[adjusted_rand_score(KMeans(3,n_init=1,random_state=a).fit(Xs).labels_, KMeans(3,n_init=1,random_state=a+100).fit(Xs).labels_) for a in range(10)]
print("n_init=1, seed-vs-seed ARI min/mean:", round(min(aris),3), round(np.mean(aris),3))
# time stability: second period with slight drift
d2 = make_shoppers(seed=9); X2 = StandardScaler().fit(df[F]).transform(d2[F])
print("next-period: assign by old centres vs refit ARI:", round(adjusted_rand_score(km.predict(X2), KMeans(3,n_init=10,random_state=0).fit(X2).labels_),3))

## 18.4 Scaffolding a Library Project: A Test Harness That Catches Your Bugs

The projects in this chapter ask you to rebuild algorithms and check them against scikit-learn. The risk is a harness so weak that wrong code passes. The remedy is a small, fixed set of checks you apply to *every* class you write, so a bug has to slip past all of them.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.datasets import make_regression
class MyRidge:
    def __init__(self, alpha=1.0): self.alpha = alpha
    def fit(self, X, y):
        self.mx, self.my = X.mean(0), y.mean(); Xc = X - self.mx
        self.coef_ = np.linalg.solve(Xc.T @ Xc + self.alpha * np.eye(X.shape[1]), Xc.T @ (y - self.my)); return self
    def predict(self, X): return (X - self.mx) @ self.coef_ + self.my
def check(name, ok): print(("PASS " if ok else "FAIL ") + name)
X, y = make_regression(200, 5, noise=3, random_state=0)
a, b = MyRidge(2.0).fit(X, y), Ridge(alpha=2.0).fit(X, y)
check("coef matches sklearn (atol 1e-8)", np.allclose(a.coef_, b.coef_, atol=1e-8))
check("predict matches sklearn", np.allclose(a.predict(X), b.predict(X), atol=1e-8))
check("alpha->inf shrinks to 0", np.abs(MyRidge(1e12).fit(X, y).coef_).max() < 1e-6)
check("row-permutation invariant", np.allclose(a.coef_, MyRidge(2.0).fit(X[::-1], y[::-1]).coef_))
bug = MyRidge(2.0); bug.fit = lambda X, y: (setattr(bug, "mx", np.zeros(5)) or setattr(bug, "my", 0.0) or setattr(bug, "coef_", np.linalg.solve(X.T @ X + 2 * np.eye(5), X.T @ y)) or bug)
check("buggy no-intercept version matches sklearn", np.allclose(bug.fit(X, y).coef_, b.coef_, atol=1e-8))

PASS coef matches sklearn (atol 1e-8)
PASS predict matches sklearn
PASS alpha->inf shrinks to 0
PASS row-permutation invariant
FAIL buggy no-intercept version matches sklearn


## Chapter recap

## Practice set

**1. (easy)** In `mlscratch`, why is the library allowed to import only NumPy? **Answer:** So that it is truly from scratch. scikit-learn is the answer key and belongs only in the tests. Acceptance: your answer mentions the separation of library and reference.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch18.html